In [2]:

# basic stuff
import torch
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
from collections import OrderedDict

# tsl
from tsl.data import SpatioTemporalDataset
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from tsl.metrics.torch import MaskedMAE, MaskedMAPE
from tsl.engines import Predictor

# pytorch lightning
from pytorch_lightning.loggers import TensorBoardLogger
import pytorch_lightning as ptl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# optuna
import optuna
# from optuna.integration import PyTorchLightningPruningCallback

# architecture from paper
from lib.nn.encoders.corel_encoder import CoRelEncoder
from lib.nn.decoder.base_decoder import BaseDecoder
from lib.nn.encoder_decoder_model import EncoderDecoderModel

# reproducibility
ptl.seed_everything(42)


Seed set to 42


42

In [ ]:
df = pd.read_csv('../data/EWZ_cleaned.csv', index_col=0)

df.head()

In [ ]:


torch_dataset_hour_forecast = SpatioTemporalDataset(target=df,
                                      horizon=4,
                                      window=4 * 24,
                                      stride=1)


# Use later on

# torch_dataset_day_forecast = SpatioTemporalDataset(target=df,
#                                       horizon=4 * 24,
#                                       window=4 * 24 * 7,
#                                       stride=1)

In [ ]:


# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
splitter = TemporalSplitter(val_len=0.1, test_len=0.25)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset_hour_forecast,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
)

input_size = torch_dataset_hour_forecast.n_channels  
n_nodes = torch_dataset_hour_forecast.n_nodes        
horizon = torch_dataset_hour_forecast.horizon         

### Hyperparameter selection with optuna

[Source](https://machinelearningmastery.com/pytorch-lightning-hyperparameter-optimization-with-optuna/)

In [ ]:
def objective(trial):

    conv_type = trial.suggest_categorical('conv_type', ['diffconv', 'graphconv'])   
    # temporal_type = trial.suggest_categorical('temporal_type', ["gru", "lstm"]) # TODO readd lstm (memory error)
    hidden_size = trial.suggest_categorical('hidden_dim', [16, 32, 64])
    k = trial.suggest_int('k', 1, 5)
    spatial_layers = trial.suggest_int('spatial_layers', 1, 2)
    temporal_layers = trial.suggest_int('temporal_layers', 1, 4)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)


    stgnn = EncoderDecoderModel(
    input_size=input_size,
    output_size=input_size,
    horizon=horizon,
    encoder_class=CoRelEncoder,
    encoder_kwargs={'gnn_layers': spatial_layers, 'temporal_layers': temporal_layers,
                    'hidden_size': hidden_size, 'n_instances': n_nodes, 'emb_size': hidden_size,
                    'n_neighbors': k, 'conv_type': conv_type, "temporal_type": "gru"},
    decoder_class=BaseDecoder,
    decoder_kwargs={},
    exog_size= 0,
    )


    

    logger = TensorBoardLogger(save_dir="logs", name=f"optuna_logs/trial_{trial.number}")

    loss_fn = MaskedMAE()

    metrics = {'mae': MaskedMAE(),}

    # setup predictor
    predictor = Predictor(
        model=stgnn,                  
        optim_class=torch.optim.Adam,  
        optim_kwargs={'lr': learning_rate},    
        loss_fn=loss_fn,               
        metrics=metrics               
    )



    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        patience=5,
        verbose=False,
        mode='min'
    )

    # Cant find work around to use it : https://github.com/Lightning-AI/pytorch-lightning/issues/17485
    # Can try to rename import in venv

    # pruning_callback = PyTorchLightningPruningCallback(trial, monitor='val_loss')


    trainer = ptl.Trainer(max_epochs=10,
                        logger=logger,
                        limit_train_batches=10,
                        callbacks=[early_stop_callback]) #, pruning_callback])
    

    trainer.fit(predictor, datamodule=dm)

    return trainer.callback_metrics['val_loss'].item()


In [ ]:
def run_optimization(n_trials=20):
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10) # should I keep this ?
    study = optuna.create_study(direction='minimize', pruner=pruner)
    study.optimize(objective, n_trials=n_trials)
    
    print("Best trial:")
    trial = study.best_trial
    print(f"  Value: {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
        
    return study

study = run_optimization()
best_params = study.best_trial.params


In [3]:
# Recover from interruption:
best_params = {'conv_type': 'diffconv', 'hidden_dim': 64, 'k': 4, 'spatial_layers': 2, 'temporal_layers': 2, 'learning_rate': 0.004739453035357187}


with open("./best_hyperparams_hour_forecast_no_lstm.json", "w") as f:
            json.dump(best_params, f)